# Fine Tune em Apple Silicon


In [2]:
!uv pip install mlx-tune mlx-lm transformers datasets torch TensorFlow ipywidgets

Using Python 3.11.14 environment at: /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/.venv
Audited 7 packages in 53ms


# Carregando o modelo base

In [ ]:
from mlx_tune import FastLanguageModel, SFTTrainer, SFTConfig
from datasets import load_dataset

max_seq_length = 8192

HF_TOKEN= "..."

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mlx-community/Llama-3.2-3B-Instruct-4bit",
    # For real Unsloth: model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit"
    max_seq_length=max_seq_length,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
    token=HF_TOKEN,
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0522 06:13:54.911000 78589 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

## Carregando adaptadores LoRA

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank: 16 is sweet spot for most tasks
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # 0 is optimized in Unsloth
    bias="none",
    # Mirrors Unsloth's tutorial syntax exactly. mlx-tune's default is
    # False (faster); pass "unsloth" or True when you need to halve
    # activation memory at the cost of ~2× step time.
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

LoRA configuration set: rank=16, alpha=16, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], dropout=0


## Definindo instrução

Vai ser necessário alterar quando tiver exemplos de conversas com contexto PCDT disponíveis e classificados.

Vai ficar mais complexo, também, quando o modelo for usado para outras tarefas, como guardrail, reescrever (ou pular) a query, etc.

In [ ]:
# TODO: quando tiver exemplos de conversa, usar o mesmo system prompt do backend
_INSTRUCTION = """\
Você é um assistente clínico de apoio a médicos no Brasil.
Responda em português do Brasil, de forma objetiva e profissional, com um leve grau de liberdade e humor leve.
"""

## Carregando dataset

In [6]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="assets/MedQuAD_excellent_ptbr.csv")

dataset

DatasetDict({
    train: Dataset({
        features: ['AnswerID', 'Answer', 'Question', 'qid', 'score', 'Pergunta_BR', 'Resposta_BR'],
        num_rows: 141
    })
})

In [29]:
def format_prompts(qa_pair):
    # texts = []
    messages = [
        {"role": "system", "content": _INSTRUCTION},
        {"role": "user", "content": qa_pair["Pergunta_BR"]},
        {"role": "assistant", "content": qa_pair["Resposta_BR"]},
    ]
    # texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    # return {"text": texts}
    return {
        "text": tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    }


format_prompts(dataset["train"][0])

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 22 May 2026\n\nVocê é um assistente clínico de apoio a médicos no Brasil.\nResponda em português do Brasil, de forma objetiva e profissional, com um leve grau de liberdade e humor leve.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nComo diagnosticar sífilis primária? (Também chamado de: Sífilis primária; Sífilis secundária; Sífilis tardia; Sífilis terciária)<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nO médico ou enfermeiro fará um exame físico. Os testes que podem ser realizados incluem: - Exame de fluido da lesão - Ecocardiograma, angiografia aórtica e cateterismo cardíaco para avaliar os grandes vasos sanguíneos e o coração - Punção lombar e exame do líquido cefalorraquidiano - Exames de sangue para rastreio de bactérias da sífilis (RPR, VDRL ou TRUST). Se os testes RPR, VDRL ou TRUST forem positivos, um dos seguintes testes será necessário

### Transformando o dataset no padrão

In [30]:
dataset = dataset.map(format_prompts)

dataset["train"][0]

Map:   0%|          | 0/141 [00:00<?, ? examples/s]

{'AnswerID': 'ADAM_0003820_Sec4.txt',
 'Answer': 'The doctor or nurse will examine you. Tests that may be done include:  - Examination of fluid from sore  - Echocardiogram, aortic angiogram, and cardiac catheterization to look at the major blood vessels and the heart  - Spinal tap and examination of spinal fluid  - Blood tests to screen for syphillis bacteria (RPR, VDRL, or TRUST)    If the RPR, VDRL, or TRUST tests are positive, one of the following tests will be needed to confirm the diagnosis:   - FTA-ABS (fluorescent treponemal antibody test)  - MHA-TP  - TP-EIA  - TP-PA)',
 'Question': 'How to diagnose Syphilis - primary ? (Also called: Primary syphilis; Secondary syphilis; Late syphilis; Tertiary syphilis)',
 'qid': 4,
 'score': '4-Excellent',
 'Pergunta_BR': 'Como diagnosticar sífilis primária? (Também chamado de: Sífilis primária; Sífilis secundária; Sífilis tardia; Sífilis terciária)',
 'Resposta_BR': 'O médico ou enfermeiro fará um exame físico. Os testes que podem ser realiz

# Fine tune em ação

## Configuração do treinamento

In [ ]:
training_config = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    # max_steps=20,  # Small for demo, use 500+ for real training
    num_train_epochs=1, # One pass through the dataset
    learning_rate=2e-4,
    # fp16=True,  # Not applicable on MLX
    # bf16=True,  # Not applicable on MLX
    logging_steps=1,
    output_dir="outputs",
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=3407,
    lr_scheduler_type="linear",
    max_length=None, # disable truncation
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    max_seq_length = max_seq_length,
    dataset_text_field = "text",
    tokenizer=tokenizer,
    packing=True, # make best use of memory by training on multiple sequences at once
    args=training_config,  # Pass SFTConfig just like in Unsloth!
)

Trainer initialized:
  Output dir: outputs
  Adapter path: outputs/adapters
  Learning rate: 0.0002
  Iterations: 70
  Batch size: 2
  LoRA r=16, alpha=16
  Native training: True
  LR scheduler: linear
  Grad checkpoint: False


In [35]:
import json
from dataclasses import asdict

print(json.dumps(asdict(training_config), indent=2))

TypeError: asdict() should be called on dataclass instances

### Execução do treinamento

In [10]:
trainer.train()

Starting Fine-Tuning

[Using Native MLX Training]

Applying LoRA adapters...
Applying LoRA to 28 layers: {'rank': 16, 'scale': 1.0, 'dropout': 0, 'keys': ['mlp.down_proj', 'mlp.gate_proj', 'mlp.up_proj', 'self_attn.k_proj', 'self_attn.o_proj', 'self_attn.q_proj', 'self_attn.v_proj']}
✓ LoRA applied successfully to 28 layers
  Trainable LoRA parameters: 392
Preparing training data...
  Detected format: text
✓ Prepared 141 training samples
  Saved to: outputs/train.jsonl
✓ Created validation set (copied from train)

Training configuration:
  Iterations: 70
  Batch size: 2
  Learning rate: 0.0002
  LR scheduler: linear
  Grad checkpoint: True
  Adapter file: outputs/adapters/adapters.safetensors

Loaded 141 training samples, 141 validation samples
Starting training loop...
Starting training..., iters: 70


Calculating loss...: 100%|██████████| 5/5 [00:12<00:00,  2.60s/it]

Iter 1: Val loss 1.969, Val took 12.995s


Iter 1: Train loss 1.687, Learning Rate 2.000e-04, It/sec 0.070, Tokens/sec 95.670, Trained Tokens 1368, Peak mem 5.260 GB
Iter 2: Train loss 2.421, Learning Rate 1.971e-04, It/sec 0.304, Tokens/sec 128.726, Trained Tokens 1792, Peak mem 5.260 GB
Iter 3: Train loss 1.315, Learning Rate 1.943e-04, It/sec 0.050, Tokens/sec 86.948, Trained Tokens 3543, Peak mem 6.211 GB
Iter 4: Train loss 1.451, Learning Rate 1.914e-04, It/sec 0.142, Tokens/sec 130.282, Trained Tokens 4461, Peak mem 6.211 GB
Iter 5: Train loss 1.228, Learning Rate 1.886e-04, It/sec 0.226, Tokens/sec 130.275, Trained Tokens 5038, Peak mem 6.211 GB
Iter 6: Train loss 1.417, Learning Rate 1.857e-04, It/sec 0.065, Tokens/sec 85.159, Trained Tokens 6350, Peak mem 6.211 GB
Iter 7: Train loss 1.240, Learning Rate 1.829e-04, It/sec 0.285, Tokens/sec 158.623, Trained Tokens 6906, Peak mem 6.211 GB
Iter 8: Train loss 1.387, Learning Rate 1.800e-04, It/sec 0.188, Tokens/sec 144.015, Trained Tokens 7671, Peak mem 6.211 GB
Iter 9: Tra

Calculating loss...: 100%|██████████| 5/5 [00:32<00:00,  6.55s/it]

Iter 70: Val loss 1.019, Val took 32.741s


Iter 70: Train loss 1.322, Learning Rate 2.857e-06, It/sec 0.023, Tokens/sec 38.753, Trained Tokens 72711, Peak mem 7.545 GB
Saved final weights to outputs/adapters/adapters.safetensors.
  Adapter config saved to: outputs/adapters/adapter_config.json

Training Complete!
  Adapters saved to: outputs/adapters


{'status': 'success', 'adapter_path': 'outputs/adapters'}

## Testando

In [17]:
from mlx_lm import generate

prompt = "O que é diabetes?"
FastLanguageModel.for_inference(model)
messages = [
    # {"role": "system", "content": _INSTRUCTION},
    {"role": "user", "content": prompt},
]
formatted_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

generate(
    model.model,
    tokenizer,
    prompt=formatted_prompt,
    max_tokens=max_seq_length,
    verbose=True,
)

Inference mode enabled with KV caching
Diabetes é uma doença crônica que afeta como o seu corpo processa a glicose (açúcar) no sangue. A glicose é um tipo de açúcar que é produzido pelo corpo e também é encontrada em muitos alimentos. O corpo usa a glicose como fonte de energia. No entanto, quando você tem diabetes, seu corpo não consegue usar a glicose corretamente. Em vez disso, a glicose se acumula no seu sangue. Se você não tiver diabetes, seu corpo pode usar a glicose para energia. Se você tiver diabetes, seu corpo não consegue usar a glicose para energia. Em vez disso, a glicose se acumula no seu sangue. Se você não tiver diabetes, seu corpo pode usar a glicose para energia. Se você tiver diabetes, seu corpo não consegue usar a glicose para energia. Em vez disso, a glicose se acumula no seu sangue.
Prompt: 40 tokens, 146.781 tokens-per-sec
Generation: 221 tokens, 47.196 tokens-per-sec
Peak memory: 7.545 GB


'Diabetes é uma doença crônica que afeta como o seu corpo processa a glicose (açúcar) no sangue. A glicose é um tipo de açúcar que é produzido pelo corpo e também é encontrada em muitos alimentos. O corpo usa a glicose como fonte de energia. No entanto, quando você tem diabetes, seu corpo não consegue usar a glicose corretamente. Em vez disso, a glicose se acumula no seu sangue. Se você não tiver diabetes, seu corpo pode usar a glicose para energia. Se você tiver diabetes, seu corpo não consegue usar a glicose para energia. Em vez disso, a glicose se acumula no seu sangue. Se você não tiver diabetes, seu corpo pode usar a glicose para energia. Se você tiver diabetes, seu corpo não consegue usar a glicose para energia. Em vez disso, a glicose se acumula no seu sangue.'

### Comparando com as respostas antes do fine tune

In [21]:
import pandas as pd

df_test = pd.read_csv("assets/MedQuAD_excellent_ptbr_respostas_antes.csv")

def format_messages_for_question(question):
    return [
        {"role": "system", "content": _INSTRUCTION},
        {"role": "user", "content": question},
    ]

def generate_or_use_existing(row):
    if pd.isna(row.get("Resposta_BR_com_ft")) or row.get("Resposta_BR_com_ft") is None:
        messages = format_messages_for_question(row["Pergunta_BR"]) 
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return generate(
            model.model, tokenizer, prompt=formatted_prompt, max_tokens=1024, verbose=True
        )
    else:
        return row["Resposta_BR_com_ft"]

df_test["Resposta_BR_com_ft"] = df_test.apply(generate_or_use_existing, axis=1)

O HPS é uma doença rara que ocorre em pessoas que não têm doenças crônicas. A doença é causada por um vírus chamado Hantavírus. O vírus é transmitido por um animal chamado ratinho. Os ratinhos são portadores do vírus. O vírus pode ser transmitido para os humanos por meio de contato direto com os ratinhos. O vírus também pode ser transmitido por meio de contato indireto com os ratinhos. O vírus pode ser transmitido por meio de contato indireto com os ratinhos. O vírus pode ser transmitido por meio de contato indireto com os ratinhos.
Prompt: 97 tokens, 455.526 tokens-per-sec
Generation: 151 tokens, 60.101 tokens-per-sec
Peak memory: 7.545 GB
O cálcio é um mineral essencial para o corpo humano. Ele ajuda a construir e manter os ossos e dentes. O cálcio também é importante para a saúde do coração e do sistema nervoso. O cálcio é encontrado em muitos alimentos, incluindo leite, queijo, iogurte, frutas, vegetais e nozes. O cálcio também é encontrado em alguns suplementos dietéticos. O cálci

In [22]:
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(df_test)

Pergunta_BR  \
0                                                                                                                   Qual é a história do HPS para Hantavírus?   
1                                                                              Você tem informações sobre Cálcio na dieta (Também chamado de: Dieta - cálcio)   
2                                                                                                                           O que é Herpes Zóster (Shingles)?   
3   Você tem informações sobre procedimentos de Ablação Cardíaca (Também chamados de: Ablação por Cateter; Ablação por Cateter Radiofrequência; Crioablação)?   
4                                                                                                                                           O que é Diabetes?   
5                                                                                               Quais são os nomes comerciais de Metilprednisolona Oral?-----   
6                                                                             Quais são os nomes comerciais de Methylprednisolone Sodium Succinate Injection?   
7                        O que fazer para secreção no ouvido? (Também chamado de: Drenagem do ouvido; Otorreia; Sangramento no ouvido; Sangramento do ouvido)   
8                                                                            Qual é o prognóstico para a Trissomia 13? (Também chamado de: Síndrome de Patau)   
9                               O que é secreção de ouvido? (Também chamado de: Drenagem do ouvido; Otorreia; Sangramento de ouvido; Sangramento pelo ouvido)   
10                       Como diagnosticar paralisia do sono isolada? (Também chamado de: Paralisia do sono - isolada; Parasônia - paralisia do sono isolada)   
11                                                                                                     Quais são os efeitos colaterais ou riscos da Metadona?   
12                                                   O que é a síndrome de Beckwith-Wiedemann? (Também chamada de: BWS; síndrome de Wiedemann-Beckwith (WBS))   
13             O que é o síndrome antifosfolípide? (Também chamado de: síndrome antifosfolípide; síndrome de anticorpos antifosfolípides; síndrome de Hughes)   
14                                                                                                                            O que causa um ataque cardíaco?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## Salvando o modelo para Ollama

In [12]:
model.save_pretrained_gguf("model", tokenizer, quantization_method="q4_k_m")

  LoRA adapters will be fused from: outputs/adapters
Exporting to GGUF format...

⚠️  WARNING: Quantized model detected!
Model 'mlx-community/Llama-3.2-3B-Instruct-4bit' appears to be quantized.
GGUF export from quantized models is NOT supported by mlx_lm.
This is an upstream limitation: https://github.com/ml-explore/mlx-lm/issues/353

Options:
  1. Use dequantize=True (creates large fp16, re-quantize with llama.cpp)
  2. Use a non-quantized base model for training
  3. Use save_pretrained_merged() for MLX-only inference

Exporting model to GGUF format...
  Model: mlx-community/Llama-3.2-3B-Instruct-4bit
  Output: model/model.gguf
  Adapters: outputs/adapters

Running: mlx_lm.fuse --model mlx-community/Llama-3.2-3B-Instruct-4bit --export-gguf --gguf-path model/model.gguf --adapter-path outputs/adapters
Error during GGUF export: 
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 72315.59it/s]
Traceback (most recent call last):
  File "/Users/leanderseefeld/workspaces/fiap/8iadt-tc-fa

usage: mlx_lm.convert [-h] [--hf-path HF_PATH] [--mlx-path MLX_PATH] [-q]
                      [--q-group-size Q_GROUP_SIZE] [--q-bits Q_BITS]
                      [--q-mode {affine,mxfp4,nvfp4,mxfp8}]
                      [--quant-predicate {mixed_2_6,mixed_3_4,mixed_3_6,mixed_4_6}]
                      [--dtype {float16,bfloat16,float32}]
                      [--upload-repo UPLOAD_REPO] [-d] [--trust-remote-code]
mlx_lm.convert: error: unrecognized arguments: --export-gguf


Alternative method also failed: Command '['mlx_lm.convert', '--hf-path', 'mlx-community/Llama-3.2-3B-Instruct-4bit', '-q', '--export-gguf']' returned non-zero exit status 2.

Manual export command:
  mlx_lm.fuse --model mlx-community/Llama-3.2-3B-Instruct-4bit --export-gguf --gguf-path model/model.gguf


CalledProcessError: Command '['mlx_lm.convert', '--hf-path', 'mlx-community/Llama-3.2-3B-Instruct-4bit', '-q', '--export-gguf']' returned non-zero exit status 2.

# IGNORE